# Extract and cache raw ($T = 1.0$) option logits

Runs a single GPU inference pass per `(model, regime)` at raw head temperature $T = 1.0$ and saves option-level logits to `analysis/post-hoc/logits_cache/<model>/<regime>/<dataset>.pt` for fast, repeatable CPU-only downstream analyses (head temperature fitting on **Flights**, calibration diagnostics, and held-out test evaluation in `04-temperature-calibration.ipynb`).

**Key properties:**
- **Bit-for-bit parity with `test_best`:** uses the exact `PopulationValidator` state-grouped encoding, `fused_eval_context` (bf16 + CUDA graphs), and `head.temperature = 1.0`, and fails fast if recomputed $T = 1.0$ betas or KL diverge from `test_best/` by more than `PARITY_BETA_ATOL` / `PARITY_KL_ATOL` (`5e-3` to allow cross-GPU `bf16` rounding between H100 and RTX 5090).
- **Resumable and atomic:** each `(model, regime, dataset).pt` file is written atomically and skipped on reruns if `checkpoint_step`, `kev_base`, `choice_fingerprint`, and `schema_version` match.
- **Single fused context per checkpoint:** enters `fused_eval_context` once per `(model, regime)` across all missing datasets to avoid repeated LoRA merge/offload and CUDA-graph capture.

In [8]:
# Environment flags, set before any Hugging Face or torch import (same as 04-test-best-*.ipynb)
import os
from pathlib import Path

ON_CLUSTER = "SLURM_JOB_ID" in os.environ
CLUSTER_HF_HOME = Path.home() / "links" / "projects" / "def-yuma" / "wadi" / "hf"
if ON_CLUSTER:
    if not CLUSTER_HF_HOME.is_dir():
        raise FileNotFoundError(f"Cluster Hugging Face cache not found: {CLUSTER_HF_HOME}")
    os.environ["HF_HOME"] = str(CLUSTER_HF_HOME)

import hashlib
import json
import pickle
import sys
import time

import numpy as np
import pandas as pd
import torch
from IPython.display import display
from tqdm.auto import tqdm

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)

# Walk up from the notebook folder to locate the repository's `code/` root
CODE_DIR = next(
    (
        c
        for p in (Path.cwd(), *Path.cwd().parents)
        for c in (p, p / "code")
        if all((c / d).is_dir() for d in ("kev", "data", "models"))
    ),
    None,
)
if CODE_DIR is None:
    raise RuntimeError(f"No code/ root containing kev/, data/, and models/ at or above {Path.cwd()}")

PAPER_DIR = CODE_DIR / "papers" / "sim-respondents"
SRC_DIR = PAPER_DIR / "src"
POST_HOC_DIR = PAPER_DIR / "analysis" / "post-hoc"

# Run from `code/` so data loaders and shared disk caches resolve identically to the training/test notebooks
os.chdir(CODE_DIR)
for _p in (str(CODE_DIR), str(CODE_DIR / "kev"), str(SRC_DIR)):
    if _p not in sys.path:
        sys.path.insert(0, _p)

from kev.checkpoint import read_meta as read_kev_meta, resolve_run as resolve_kev_run
from kev.cuda_graphs import Request
from kev.model import EAGER_STATES, rows_of
from data.base import DEFAULT_CACHE_DIR, fused_eval_context, group_eval_views
from models.kev_pointer import KevPointerModel
from training import free_gpu

import beta_ols
from beta_ols.evaluation import compare_population_betas, flat_probs_from_predictions
from beta_train import runs as run_store
from beta_train.model_state import reset_trainable, set_checkpointing, snapshot_trainable
from beta_train.validation import PopulationValidator

print(f"CODE_DIR: {CODE_DIR}")
print(f"CUDA available: {torch.cuda.is_available()}")

CODE_DIR: /home/dw/github/decision-models-finetuning/code
CUDA available: True


In [9]:
# ==============================================================================
# Configuration
# ==============================================================================
SCHEMA_VERSION = 1
HEAD_TEMPERATURE = 1.0
LORA_RANK = 64
WEIGHTS_DTYPE = "bf16"
VALIDATION_MAX_QUESTIONS = 1024
VALIDATION_MAX_CHARS = 1_000_000
PARITY_BETA_ATOL = 5e-3
PARITY_KL_ATOL = 5e-3

OUTPUT_DIR = POST_HOC_DIR / "logits_cache"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Exact pinned checkpoints and run roots matching 04-test-best-*.ipynb
MODEL_SPECS = {
    "kev-0.8b": {
        "label": "KEV-0.8B",
        "kev_base": "jaredpalmer/kev-0.8b@bf75a6a8848ea6960ff2ed108d9ed44c2941174f",
        "runs_dirs": [PAPER_DIR / "runs-rorqual" / "kev-0.8b"],
    },
    "kev-4b": {
        "label": "KEV-4B",
        "kev_base": "jaredpalmer/kev-4b@6cfce5c2fa4b4bd64026336ab649c5ca78857d52",
        "runs_dirs": [PAPER_DIR / "runs-rorqual" / "kev-4b"],
    },
    "kev-9b": {
        "label": "KEV-9B",
        "kev_base": "jaredpalmer/kev-9b@db029f08b290afd9fee4aa4bbcd9ae48602d1eb0",
        "runs_dirs": [
            PAPER_DIR / "runs-rorqual" / "kev-9b",
            PAPER_DIR / "runs-nibi" / "kev-9b",
        ],
    },
}

REGIME_LABELS = {
    "row1_base_untrained": "Off-the-shelf",
    "row2b_wu_ce_oracle": "Electricity",
    "row3_berke_ce_only": "Drones",
    "row5_full_berke_wu_ce_oracle": "Drones + Electricity",
}

DATASET_LABELS = {
    "kriner2026": "Flights",
    "kantorowicz2026": "Consumer rights",
    "magistro2025": "AI & offshoring",
    "grummon2025": "Food labels",
    "covid": "Vaccines",
}

# Default selection: all 4 paper regimes (Off-the-shelf, Electricity, Drones, Drones + Electricity)
# across all 3 sizes and 5 datasets (override via LOGITS_MODELS / LOGITS_REGIMES / LOGITS_DATASETS).
DEFAULT_MODELS = ["kev-0.8b", "kev-4b", "kev-9b"]
DEFAULT_REGIMES = [
    "row1_base_untrained",
    "row2b_wu_ce_oracle",
    "row3_berke_ce_only",
    "row5_full_berke_wu_ce_oracle",
]
DEFAULT_DATASETS = ["kriner2026", "kantorowicz2026", "magistro2025", "grummon2025", "covid"]

def _parse_env_list(var_name: str, default: list[str], allowed: set[str]) -> list[str]:
    if var_name not in os.environ or not os.environ[var_name].strip():
        return list(default)
    raw = os.environ[var_name].strip()
    items = [s.strip() for s in raw.split(",") if s.strip()]
    unknown = [x for x in items if x not in allowed]
    if unknown:
        raise ValueError(f"{var_name} contains unknown entries {unknown}; allowed: {sorted(allowed)}")
    return items

MODELS = _parse_env_list("LOGITS_MODELS", DEFAULT_MODELS, set(MODEL_SPECS))
REGIMES = _parse_env_list("LOGITS_REGIMES", DEFAULT_REGIMES, set(REGIME_LABELS))
DATASETS = _parse_env_list("LOGITS_DATASETS", DEFAULT_DATASETS, set(DATASET_LABELS))

print("Selected MODELS  :", MODELS)
print("Selected REGIMES :", REGIMES)
print("Selected DATASETS:", DATASETS)
print("OUTPUT_DIR       :", OUTPUT_DIR)

Selected MODELS  : ['kev-0.8b', 'kev-4b', 'kev-9b']
Selected REGIMES : ['row1_base_untrained', 'row2b_wu_ce_oracle', 'row3_berke_ce_only', 'row5_full_berke_wu_ce_oracle']
Selected DATASETS: ['kriner2026', 'kantorowicz2026', 'magistro2025', 'grummon2025', 'covid']
OUTPUT_DIR       : /home/dw/github/decision-models-finetuning/code/papers/sim-respondents/analysis/post-hoc/logits_cache


## 1. Build dataset validators (CPU)

In [10]:
validators: dict[str, PopulationValidator] = {}
for dataset_name in DATASETS:
    dataset_module = beta_ols.DATASET_MODULES[dataset_name]
    choice_df = dataset_module.load_choice_df()
    rows = dataset_module.regression_rows(choice_df)
    specs = dataset_module.regression_specs(rows)
    validators[dataset_name] = PopulationValidator(
        dataset_module,
        choice_df,
        specs,
        max_questions=VALIDATION_MAX_QUESTIONS,
        max_chars=VALIDATION_MAX_CHARS,
    )

val_summary = pd.DataFrame(
    [
        {
            "dataset": ds,
            "label": DATASET_LABELS[ds],
            "role": "Validation" if ds == "kriner2026" else "Test",
            "tasks": len(v.bundle["records"]),
            "total_options": int(v.eval_ops["total_options"]),
            "packed_states": len(v.bundle["packed_views"]),
            "n_betas": int(sum(len(s.target_labels) for s in v.eval_ops["specs"].values())),
            "prep_sec": round(v.prep_sec, 2),
        }
        for ds, v in validators.items()
    ]
).set_index("dataset")
display(val_summary)

,label,role,tasks,total_options,packed_states,n_betas,prep_sec
dataset,,,,,,,
kriner2026,Flights,Validation,5000,10000,990,12,0.83
kantorowicz2026,Consumer rights,Test,26823,53646,42,38,3.81
magistro2025,AI & offshoring,Test,21005,105025,2221,32,4.12
grummon2025,Food labels,Test,33798,101394,4906,5,7.13
covid,Vaccines,Test,9855,29565,1845,11,1.55


## 2. Helper functions (checkpoint resolution, fused raw-logit forward, parity verification, atomic cache)

In [11]:
def resolve_run_dir(model_slug: str, regime_slug: str) -> Path:
    runs_dirs = MODEL_SPECS[model_slug]["runs_dirs"]
    if regime_slug == "row1_base_untrained":
        candidate = runs_dirs[0] / "row1_base_untrained"
        if not (candidate / "test_best" / "test.json").exists():
            raise FileNotFoundError(f"Missing Row 1 test_best directory: {candidate}")
        return candidate
    found = [d / regime_slug for d in runs_dirs if (d / regime_slug / "run.json").exists()]
    if len(found) != 1:
        raise FileNotFoundError(
            f"Expected exactly 1 run directory for {model_slug}/{regime_slug} across {runs_dirs}, found {found}"
        )
    return found[0]


def resolve_checkpoint_spec(model_slug: str, regime_slug: str) -> dict:
    run_dir = resolve_run_dir(model_slug, regime_slug)
    if regime_slug == "row1_base_untrained":
        return {
            "run_dir": run_dir,
            "paper_row": "Row 1",
            "checkpoint_step": 0,
            "is_trained": False,
        }
    run_info = run_store.read_run_json(run_dir)
    if run_info["status"] != run_store.STATUS_FINISHED:
        raise RuntimeError(
            f"{model_slug}/{regime_slug} status is {run_info['status']!r}, expected {run_store.STATUS_FINISHED!r}"
        )
    test_json = run_dir / "test_best" / "test.json"
    if not test_json.exists():
        raise FileNotFoundError(f"Missing test_best/test.json in {run_dir}")
    test_info = json.loads(test_json.read_text(encoding="utf-8"))
    best_ckpt_path = run_dir / "checkpoints" / "best.pt"
    if not best_ckpt_path.exists():
        raise FileNotFoundError(f"Missing best.pt checkpoint: {best_ckpt_path}")
    return {
        "run_dir": run_dir,
        "paper_row": str(run_info["identity"]["run_metadata"]["paper_row"]),
        "checkpoint_step": int(test_info["checkpoint_step"]),
        "is_trained": True,
    }


def get_shipped_temperature(kev_base: str) -> float:
    meta = read_kev_meta(resolve_kev_run(kev_base))
    return float(meta.temperature)


def cache_file_path(model_slug: str, regime_slug: str, dataset_name: str) -> Path:
    return OUTPUT_DIR / model_slug / regime_slug / f"{dataset_name}.pt"


def expected_cache_meta(model_slug: str, regime_slug: str, dataset_name: str, ckpt_spec: dict) -> dict:
    return {
        "schema_version": SCHEMA_VERSION,
        "model_slug": model_slug,
        "kev_base": MODEL_SPECS[model_slug]["kev_base"],
        "regime_slug": regime_slug,
        "checkpoint_step": int(ckpt_spec["checkpoint_step"]),
        "dataset": dataset_name,
        "choice_fingerprint": validators[dataset_name].choice_fingerprint,
        "head_temperature": float(HEAD_TEMPERATURE),
        "lora_rank": int(LORA_RANK),
        "weights_dtype": WEIGHTS_DTYPE,
    }


def is_cached_valid(out_path: Path, expected: dict) -> bool:
    if not out_path.exists():
        return False
    payload = torch.load(out_path, map_location="cpu", weights_only=False)
    meta = payload["meta"]
    return all(meta[k] == v for k, v in expected.items())


def save_logits_atomic(out_path: Path, payload: dict) -> None:
    out_path.parent.mkdir(parents=True, exist_ok=True)
    tmp_path = out_path.with_name(f"{out_path.name}.{os.getpid()}.tmp")
    try:
        torch.save(payload, tmp_path)
        tmp_path.replace(out_path)
    except BaseException:
        tmp_path.unlink(missing_ok=True)
        raise


def prepare_dataset_encs(kev_model: KevPointerModel, validator: PopulationValidator) -> list:
    """Tokenize packed persona views identically to data.base.predict_probs_grouped (reusing disk cache)."""
    bundle = validator.bundle
    packed_views = bundle["packed_views"]
    label = bundle["name"]
    encs_key = (kev_model.tok.name_or_path, kev_model.encoder.option_isolation, len(packed_views))
    if bundle["encs"] is not None and bundle["encs_key"] == encs_key:
        return bundle["encs"]
    h = hashlib.sha256()
    h.update(f"tok={kev_model.tok.name_or_path}|iso={kev_model.encoder.option_isolation}|n={len(packed_views)}".encode())
    for view in packed_views:
        h.update(json.dumps(view, sort_keys=True).encode())
    cache_path = DEFAULT_CACHE_DIR / f"encs_{label}_{h.hexdigest()[:12]}.pkl"
    if cache_path.exists():
        encs = pickle.loads(cache_path.read_bytes())
    else:
        encs = [kev_model.prepare(view)[0] for view in packed_views]
        DEFAULT_CACHE_DIR.mkdir(parents=True, exist_ok=True)
        tmp_path = cache_path.with_name(f"{cache_path.name}.{os.getpid()}.tmp")
        tmp_path.write_bytes(pickle.dumps(encs, protocol=pickle.HIGHEST_PROTOCOL))
        tmp_path.replace(cache_path)
    bundle["encs"] = encs
    bundle["encs_key"] = encs_key
    return encs


@torch.no_grad()
def logits_batch(net, encs: list) -> list[list[torch.Tensor]]:
    """Raw T=1.0 option logits via the exact fused CUDA-graph path used by DecisionModel.probs_batch."""
    if net.training:
        raise RuntimeError("Raw logit extraction requires model.eval() mode")
    if net.head.temperature != 1.0:
        raise RuntimeError(f"Raw logit extraction requires head.temperature == 1.0, got {net.head.temperature}")
    splits = [rows_of(e) for e in encs]

    def fits(i: int, cached: bool) -> bool:
        S, _, rows = splits[i]
        return net.graphs is not None and net.graphs.admits(len(S), [len(r["ids"]) for r in rows], cached)

    long = {i for i in range(len(encs)) if not fits(i, False) and fits(i, True)}
    batched = [i for i in range(len(encs)) if i in long or fits(i, False)]
    out: dict[int, list[torch.Tensor]] = {}
    for i in sorted(set(range(len(encs))) - set(batched)):
        out[i] = [z.float().cpu() for z in net.forward(encs[i])]

    runs: list[list[int]] = [[]]
    held = 0
    for i in batched:
        if i in long and held == EAGER_STATES:
            runs.append([])
            held = 0
        runs[-1].append(i)
        held += int(i in long)

    for run in filter(None, runs):
        pre = {i: net.prefix(encs[i]) if i in long else None for i in run}
        reqs = [
            Request(
                splits[i][0],
                splits[i][1],
                [(r["ids"], r["pos"]) for r in splits[i][2]],
                None if pre[i] is None else pre[i][1],
                False,
                [[r["decide"], *r["opts"]] for r in splits[i][2]],
            )
            for i in run
        ]
        X, _ = net.graphs.run(reqs)
        ks = [len(r["opts"]) for i in run for r in splits[i][2]]
        owner = [q for q, k in enumerate(ks) for _ in range(k)]
        starts = [0]
        for k in ks:
            starts.append(starts[-1] + 1 + k)
        dec = torch.tensor(starts[:-1], device=net.device)
        opt = torch.tensor([starts[q] + 1 + j for q, k in enumerate(ks) for j in range(k)], device=net.device)
        own = torch.tensor(owner, device=net.device)
        z_split = net.head.many(X[dec], X[opt], own).float().cpu().split(ks)
        zs = iter(z_split)
        for i in run:
            out[i] = [next(zs) for _ in splits[i][2]]
    return [out[i] for i in range(len(encs))]


@torch.no_grad()
def extract_dataset_logits(
    kev_model: KevPointerModel,
    validator: PopulationValidator,
    encs: list,
    progress_desc: str | None = None,
) -> tuple[list[torch.Tensor], float]:
    """Run one dataset inside an already-active fused_eval_context and unpack to bundle['records'] order."""
    packed_views = validator.bundle["packed_views"]
    batches = group_eval_views(
        packed_views,
        max_questions=validator.max_questions,
        max_chars=validator.max_chars,
    )
    total_tasks = len(validator.bundle["records"])
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    packed_logits: list[list[torch.Tensor] | None] = [None] * len(encs)
    with tqdm(
        total=total_tasks,
        desc=progress_desc or validator.bundle["name"],
        unit="task",
        dynamic_ncols=True,
        leave=False,
    ) as pbar:
        for batch_num, batch_indices in enumerate(batches, start=1):
            b_encs = [encs[i] for i in batch_indices]
            z_batch = logits_batch(kev_model.net, b_encs)
            if kev_model.net.graphs.capture_due(False):
                kev_model.net.graphs.capture_pending()
            batch_tasks = 0
            for orig_idx, q_list in zip(batch_indices, z_batch, strict=True):
                packed_logits[orig_idx] = q_list
                batch_tasks += len(q_list)
            pbar.set_postfix(batch=f"{batch_num}/{len(batches)}", refresh=False)
            pbar.update(batch_tasks)
    torch.cuda.synchronize()
    eval_wall_sec = time.perf_counter() - t0

    rec_logits: list[torch.Tensor] = []
    for rec_idx, row_map in enumerate(validator.bundle["unpack_idx"]):
        if len(row_map) != 1:
            raise ValueError(f"Expected 1 question per record in unpack_idx[{rec_idx}], got {len(row_map)}")
        s_idx, q_pos = row_map[0]
        z = packed_logits[s_idx][q_pos]
        expected_k = validator.eval_ops["record_positions"][rec_idx][1]
        if z.ndim != 1 or int(z.shape[0]) != expected_k:
            raise ValueError(f"Record {rec_idx}: expected {expected_k} logits, got shape {tuple(z.shape)}")
        if not bool(torch.isfinite(z).all()):
            raise ValueError(f"Record {rec_idx}: non-finite logits encountered")
        rec_logits.append(z)
    return rec_logits, eval_wall_sec


def verify_t1_parity(
    dataset_name: str,
    rec_logits: list[torch.Tensor],
    validator: PopulationValidator,
    run_dir: Path,
    checkpoint_step: int,
) -> dict:
    """Compute T=1.0 metrics and assert exact parity against test_best/ for held-out test datasets."""
    preds = [torch.softmax(z.to(torch.float64), dim=-1).numpy() for z in rec_logits]
    chosen_log_probs = [
        float(np.log(max(float(p[c]), 1e-12)))
        for p, c in zip(preds, validator.human_choices, strict=True)
    ]
    kl_loss = -float(np.mean(chosen_log_probs))
    probs_flat = flat_probs_from_predictions(validator.eval_ops, preds)
    beta_metrics, beta_df = compare_population_betas(validator.eval_ops, probs_flat)
    result = {
        "kl_loss": kl_loss,
        "beta_mse": float(beta_metrics["beta_mse"]),
        "beta_mae": float(beta_metrics["beta_mae"]),
        "beta_mape_pct": float(beta_metrics["beta_mape_pct"]),
        "spearman_rho": float(beta_metrics["spearman_rho"]),
        "parity_max_beta_diff": None,
        "parity_kl_diff": None,
    }

    if dataset_name == "kriner2026":
        return result

    test_dir = run_dir / "test_best"
    test_info = json.loads((test_dir / "test.json").read_text(encoding="utf-8"))
    if int(test_info["checkpoint_step"]) != checkpoint_step:
        raise ValueError(
            f"test_best step mismatch in {test_dir}: {test_info['checkpoint_step']} != {checkpoint_step}"
        )
    ref_metrics = pd.read_csv(test_dir / "metrics.csv").set_index("dataset")
    ref_betas = pd.read_csv(test_dir / "betas.csv")
    ref_ds = ref_betas[ref_betas["validator"] == dataset_name].set_index("term")
    if list(ref_ds.index) != list(beta_df.index):
        raise ValueError(f"{dataset_name}: beta term order mismatch vs test_best/betas.csv")
    max_beta_diff = float((beta_df["model_pop_beta"] - ref_ds["model_pop_beta"]).abs().max())
    kl_diff = abs(kl_loss - float(ref_metrics.loc[dataset_name, "kl_loss"]))
    if max_beta_diff > PARITY_BETA_ATOL or kl_diff > PARITY_KL_ATOL:
        raise AssertionError(
            f"Parity check failed for {run_dir.name}/{dataset_name}: "
            f"max_beta_diff={max_beta_diff:.3e} (atol={PARITY_BETA_ATOL}), "
            f"kl_diff={kl_diff:.3e} (atol={PARITY_KL_ATOL})"
        )
    result["parity_max_beta_diff"] = max_beta_diff
    result["parity_kl_diff"] = kl_diff
    return result


# Pre-resolve every requested (model, regime) checkpoint before touching the GPU
ckpt_specs: dict[tuple[str, str], dict] = {}
plan_rows = []
for model_slug in MODELS:
    shipped_t = get_shipped_temperature(MODEL_SPECS[model_slug]["kev_base"])
    for regime_slug in REGIMES:
        spec = resolve_checkpoint_spec(model_slug, regime_slug)
        spec["shipped_temperature"] = shipped_t
        ckpt_specs[(model_slug, regime_slug)] = spec
        missing = [
            ds
            for ds in DATASETS
            if not is_cached_valid(
                cache_file_path(model_slug, regime_slug, ds),
                expected_cache_meta(model_slug, regime_slug, ds, spec),
            )
        ]
        plan_rows.append(
            {
                "model": MODEL_SPECS[model_slug]["label"],
                "regime": REGIME_LABELS[regime_slug],
                "step": spec["checkpoint_step"],
                "shipped_T": round(shipped_t, 3),
                "cached": len(DATASETS) - len(missing),
                "to_extract": len(missing),
                "missing_datasets": ", ".join(missing) if missing else "(all cached)",
            }
        )
display(pd.DataFrame(plan_rows))

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

,model,regime,step,shipped_T,cached,to_extract,missing_datasets
0,KEV-0.8B,Off-the-shelf,0,2.351,5,0,(all cached)
1,KEV-0.8B,Electricity,640,2.351,5,0,(all cached)
2,KEV-0.8B,Drones,1152,2.351,5,0,(all cached)
3,KEV-0.8B,Drones + Electricity,1600,2.351,5,0,(all cached)
4,KEV-4B,Off-the-shelf,0,2.406,5,0,(all cached)
5,KEV-4B,Electricity,416,2.406,5,0,(all cached)
6,KEV-4B,Drones,544,2.406,5,0,(all cached)
7,KEV-4B,Drones + Electricity,1472,2.406,5,0,(all cached)
8,KEV-9B,Off-the-shelf,0,2.194,5,0,(all cached)
9,KEV-9B,Electricity,928,2.194,0,5,"kriner2026, kantorowicz2026, magistro2025, gru..."


## 3. Extract and save missing raw logits (GPU)

In [12]:
for model_slug in MODELS:
    model_cfg = MODEL_SPECS[model_slug]
    missing_by_regime: dict[str, list[str]] = {}
    for regime_slug in REGIMES:
        spec = ckpt_specs[(model_slug, regime_slug)]
        missing = [
            ds
            for ds in DATASETS
            if not is_cached_valid(
                cache_file_path(model_slug, regime_slug, ds),
                expected_cache_meta(model_slug, regime_slug, ds, spec),
            )
        ]
        if missing:
            missing_by_regime[regime_slug] = missing

    if not missing_by_regime:
        print(f"[{model_cfg['label']}] All requested regimes/datasets already cached; skipping model load.")
        continue

    if not torch.cuda.is_available():
        raise RuntimeError("CUDA GPU is required to extract uncached logits")

    free_gpu()
    t_load = time.perf_counter()
    kev_model = KevPointerModel.build(
        model_cfg["kev_base"],
        device="cuda",
        lora_rank=LORA_RANK,
        weights_dtype=WEIGHTS_DTYPE,
    )
    kev_model.net.head.temperature = HEAD_TEMPERATURE
    set_checkpointing(kev_model, False)
    kev_model.eval()
    init_trainable_state = snapshot_trainable(kev_model)
    print(f"[{model_cfg['label']}] Loaded backbone in {time.perf_counter() - t_load:.1f}s")

    # Tokenize each needed dataset once for this model's tokenizer
    needed_datasets = sorted({ds for ds_list in missing_by_regime.values() for ds in ds_list})
    encs_by_ds = {ds: prepare_dataset_encs(kev_model, validators[ds]) for ds in needed_datasets}

    try:
        for regime_slug, missing_ds in missing_by_regime.items():
            spec = ckpt_specs[(model_slug, regime_slug)]
            if spec["is_trained"]:
                checkpoint = run_store.load_best(spec["run_dir"])
                actual_step = int(checkpoint["best"]["step"])
                if actual_step != spec["checkpoint_step"]:
                    raise ValueError(
                        f"{model_slug}/{regime_slug}: best.pt step {actual_step} != test.json step {spec['checkpoint_step']}"
                    )
                reset_trainable(kev_model, checkpoint["trainable"])
                del checkpoint
            else:
                reset_trainable(kev_model, init_trainable_state)

            kev_model.net.head.temperature = HEAD_TEMPERATURE
            kev_model.eval()
            free_gpu()
            print(f"\n[{model_cfg['label']} | {REGIME_LABELS[regime_slug]} | step {spec['checkpoint_step']}] "
                  f"Extracting {missing_ds}...")

            # Single fused_eval_context across all missing datasets for this checkpoint
            graph_limits = validators[missing_ds[0]].graph_limits
            with torch.no_grad(), torch.autocast("cuda", dtype=torch.bfloat16), fused_eval_context(
                kev_model, eval_dtype=torch.bfloat16, graph_limits=graph_limits
            ):
                for ds_idx, dataset_name in enumerate(missing_ds, start=1):
                    validator = validators[dataset_name]
                    desc = (
                        f"[{ds_idx}/{len(missing_ds)}] "
                        f"{model_cfg['label']} | {REGIME_LABELS[regime_slug]} | {dataset_name}"
                    )
                    rec_logits, eval_wall_sec = extract_dataset_logits(
                        kev_model,
                        validator,
                        encs_by_ds[dataset_name],
                        progress_desc=desc,
                    )
                    t1_metrics = verify_t1_parity(
                        dataset_name,
                        rec_logits,
                        validator,
                        spec["run_dir"],
                        spec["checkpoint_step"],
                    )
                    logits_flat = torch.cat(rec_logits, dim=0).to(torch.float32).contiguous()
                    option_counts = torch.tensor([int(z.shape[0]) for z in rec_logits], dtype=torch.int32)
                    human_choices = torch.tensor(validator.human_choices, dtype=torch.int32)
                    record_positions = torch.tensor(validator.eval_ops["record_positions"], dtype=torch.int32)
                    caseids = [str(r["CASEID"]) for r in validator.bundle["records"]]
                    qids = [str(r["qid"]) for r in validator.bundle["records"]]

                    meta = {
                        **expected_cache_meta(model_slug, regime_slug, dataset_name, spec),
                        "model_label": model_cfg["label"],
                        "regime_label": REGIME_LABELS[regime_slug],
                        "paper_row": spec["paper_row"],
                        "dataset_label": DATASET_LABELS[dataset_name],
                        "shipped_temperature": float(spec["shipped_temperature"]),
                        "n_tasks": int(len(rec_logits)),
                        "total_options": int(logits_flat.shape[0]),
                        "eval_wall_sec": float(eval_wall_sec),
                        "t1_metrics": t1_metrics,
                    }
                    payload = {
                        "meta": meta,
                        "logits_flat": logits_flat,
                        "option_counts": option_counts,
                        "human_choices": human_choices,
                        "record_positions": record_positions,
                        "caseids": caseids,
                        "qids": qids,
                    }
                    out_path = cache_file_path(model_slug, regime_slug, dataset_name)
                    save_logits_atomic(out_path, payload)
                    parity_note = (
                        f"parity max|dBeta|={t1_metrics['parity_max_beta_diff']:.2e}"
                        if t1_metrics["parity_max_beta_diff"] is not None
                        else "validation dataset"
                    )
                    print(
                        f"  {dataset_name:16s} | {eval_wall_sec:6.1f}s | "
                        f"KL={t1_metrics['kl_loss']:.5f} | MSE_beta={t1_metrics['beta_mse']:.6f} | "
                        f"{parity_note} -> {out_path.relative_to(POST_HOC_DIR)}"
                    )
            free_gpu()
    finally:
        reset_trainable(kev_model, init_trainable_state)
        del kev_model, init_trainable_state
        free_gpu()

[KEV-0.8B] All requested regimes/datasets already cached; skipping model load.
[KEV-4B] All requested regimes/datasets already cached; skipping model load.


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/427 [00:00<?, ?it/s]

... (19 outputs left out)

[4/5] KEV-9B | Drones | grummon2025:   0%|          | 0/33798 [00:00<?, ?task/s]

  grummon2025      |  583.1s | KL=0.96291 | MSE_beta=0.010903 | parity max|dBeta|=9.20e-10 -> logits_cache/kev-9b/row3_berke_ce_only/grummon2025.pt


[5/5] KEV-9B | Drones | covid:   0%|          | 0/9855 [00:00<?, ?task/s]

  covid            |  266.7s | KL=1.16474 | MSE_beta=0.009536 | parity max|dBeta|=4.59e-10 -> logits_cache/kev-9b/row3_berke_ce_only/covid.pt


## 4. Inventory of cached raw logits on disk

In [7]:
inventory_rows = []
for pt_path in sorted(OUTPUT_DIR.glob("*/*/*.pt")):
    payload = torch.load(pt_path, map_location="cpu", weights_only=False)
    meta = payload["meta"]
    t1 = meta["t1_metrics"]
    inventory_rows.append(
        {
            "model": meta["model_label"],
            "regime": meta["regime_label"],
            "dataset": meta["dataset_label"],
            "step": meta["checkpoint_step"],
            "tasks": meta["n_tasks"],
            "options": meta["total_options"],
            "size_mb": round(pt_path.stat().st_size / (1024 ** 2), 2),
            "t1_kl": round(t1["kl_loss"], 5),
            "t1_beta_mse_x1e3": round(t1["beta_mse"] * 1e3, 3),
            "parity_max_beta_diff": t1["parity_max_beta_diff"],
            "wall_sec": round(meta["eval_wall_sec"], 1),
        }
    )
inventory_df = pd.DataFrame(inventory_rows)
print(f"Total cached logit files: {len(inventory_df)} ({sum(p.stat().st_size for p in OUTPUT_DIR.glob('*/*/*.pt')) / (1024**2):.2f} MB)")
display(inventory_df)

Total cached logit files: 50 (107.10 MB)


,model,regime,dataset,step,tasks,options,size_mb,t1_kl,t1_beta_mse_x1e3,parity_max_beta_diff,wall_sec
0,KEV-0.8B,Off-the-shelf,Vaccines,0,9855,29565,0.79,1.18456,12.073,1.369971e-09,28.0
1,KEV-0.8B,Off-the-shelf,Food labels,0,33798,101394,3.98,1.05890,5.463,1.132792e-09,63.3
2,KEV-0.8B,Off-the-shelf,Consumer rights,0,26823,53646,3.21,1.41834,8.339,7.723922e-10,67.1
3,KEV-0.8B,Off-the-shelf,Flights,0,5000,10000,0.46,1.08483,17.747,NaN,8.9
4,KEV-0.8B,Off-the-shelf,AI & offshoring,0,21005,105025,2.26,2.28346,33.918,1.282250e-08,51.6
5,KEV-0.8B,Electricity,Vaccines,640,9855,29565,0.79,1.08210,3.728,1.667071e-09,29.3
6,KEV-0.8B,Electricity,Food labels,640,33798,101394,3.98,0.91534,3.533,1.906803e-09,66.0
7,KEV-0.8B,Electricity,Consumer rights,640,26823,53646,3.21,0.82102,7.352,6.092758e-10,69.4
8,KEV-0.8B,Electricity,Flights,640,5000,10000,0.46,0.69304,8.917,NaN,8.3
9,KEV-0.8B,Electricity,AI & offshoring,640,21005,105025,2.26,1.68919,29.602,1.155841e-08,52.1
